In [ ]:
import asyncio
import random
import datetime
import redis.asyncio as redis
import nest_asyncio

nest_asyncio.apply()

num_test_streams = 3
pub_freq = 1
stream_max_len = 100

async def publish_test_data_for_stream(stream_index, redis_client):
    last_price = 100.0  # Starting price
    stream_key = f"test_{stream_index}"  # Using test_1, test_2, etc.

    while True:
        # Simulate large swings by adding more volatility
        change = random.uniform(-5, 5)  # Increased fluctuation range
        last_price = max(10, last_price + change)  # Keep price above zero

        # Force RSI boundary conditions sometimes
        if random.random() < 0.1:  
            last_price *= random.choice([0.85, 1.15])  # Big jumps 15% up or down

        # Create fake OHLC data
        data = {
            "symbol": "TEST",
            "timestamp": datetime.datetime.now(datetime.UTC).isoformat(),
            "open": round(last_price - random.uniform(0.5, 2), 2),
            "high": round(last_price + random.uniform(0.5, 2), 2),
            "low": round(last_price - random.uniform(1, 3), 2),
            "close": round(last_price, 2),
            "volume": random.randint(100, 1000),
            "trade_count": random.randint(10, 50),
            "vwap": round(last_price + random.uniform(-1, 1), 2),
        }

        # Push data to Redis stream
        await redis_client.xadd(stream_key, data, maxlen=stream_max_len)
        print(f"Pushed to {stream_key}: {data}")

        await asyncio.sleep(pub_freq)  # Adjust frequency if needed

async def publish_test_data(num_streams=1):
    redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

    # Create a list of tasks to run multiple streams concurrently
    tasks = []
    for stream_index in range(1, num_streams + 1):
        task = asyncio.create_task(publish_test_data_for_stream(stream_index, redis_client))
        tasks.append(task)

    # Run all the tasks concurrently
    await asyncio.gather(*tasks)


await publish_test_data(num_streams=num_test_streams)

Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-29T01:33:48.738166+00:00', 'open': 95.99, 'high': 98.27, 'low': 93.66, 'close': 96.55, 'volume': 553, 'trade_count': 49, 'vwap': 95.77}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-29T01:33:48.738166+00:00', 'open': 98.81, 'high': 101.41, 'low': 98.44, 'close': 99.53, 'volume': 868, 'trade_count': 29, 'vwap': 99.83}
Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-29T01:33:48.738166+00:00', 'open': 94.7, 'high': 97.46, 'low': 93.65, 'close': 95.8, 'volume': 577, 'trade_count': 10, 'vwap': 96.52}
Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-29T01:33:49.758844+00:00', 'open': 98.39, 'high': 101.61, 'low': 98.62, 'close': 99.92, 'volume': 516, 'trade_count': 17, 'vwap': 100.57}
Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-29T01:33:49.758844+00:00', 'open': 97.83, 'high': 99.1, 'low': 96.01, 'close': 98.4, 'volume': 987, 'trade_count': 50, 'vwap': 97.84}
Pushed to test_3: {'symbol